c:\Users\ali\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ali\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ali\.cache\huggingface\hub\datasets--AI-Lab-Makerere--beans. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an admini

In [5]:
import os
from datasets import load_dataset

ds = load_dataset("AI-Lab-Makerere/beans")

BASE_DIR = "./dataset"

for split in ds:
    for i, item in enumerate(ds[split]):
        label = ds[split].features["labels"].names[item["labels"]]
        out_dir = os.path.join(BASE_DIR, split, label)
        os.makedirs(out_dir, exist_ok=True)

        item["image"].save(os.path.join(out_dir, f"{i}.jpg"))

print("Saved to /content/beans_data")


Saved to /content/beans_data


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim


In [6]:
from torch.utils.data import DataLoader
from torchvision import datasets,transforms,models


In [ ]:

import os

In [8]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [9]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
]
)

In [10]:
trains_ds = datasets.ImageFolder("/content/beans_data/train",transform=transform)
test_ds = datasets.ImageFolder("/content/beans_data/test",transform=transform)

In [11]:
train_loader = DataLoader(trains_ds,batch_size=32,shuffle=True)
test_loader = DataLoader(test_ds,batch_size=32,shuffle=True)

In [12]:

model = models.resnet18(pretrained=True)
for params in model.parameters():
  params.requires_grade = False
model.fc = nn.Linear(model.fc.in_features,3)
model.fc.require_grad = True
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 175MB/s]


In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001,)

In [14]:
len(train_loader.dataset)

1034

In [15]:
running_loss = 0.0
for images,labels in train_loader:
  images , labels = images.to(device),labels.to(device)
  optimizer.zero_grad()
  output = model(images)
  loss = criterion(output,labels)
  running_loss += loss.item() * images.size(0)

epoch_loss = running_loss / len(train_loader.dataset)
print(f"Epoch Loss: {epoch_loss:.4f}")











Epoch Loss: 1.1595


In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
  for images ,labels in train_loader:
    images,labels= images.to(device),labels.to(device)
    output = model(images)
    loss = criterion(output,labels)
    _,predic = torch.max(output,1)
    correct += (predic == labels).sum().item()
    
    total  += labels.size(0)
val_acc = correct / total
print(f"Validation Accuracy: {val_acc:.4f}")


Validation Accuracy: 0.3366


In [17]:
from tqdm import tqdm

In [ ]:

num_epochs = 50

for epoch in range(num_epochs):
    
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    
    model.eval()
    correct = 0
    total = 0
    for images, labels in tqdm(test_loader, desc=f"Validation Epoch {epoch+1}"):
        images, labels = images.to(device), labels.to(device)
        with torch.no_grad():
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f} Val Acc: {val_acc:.4f}")


Training Epoch 1:  58%|█████▊    | 19/33 [00:04<00:03,  4.30it/s]

In [ ]:
torch.save(model.state_dict(), "/content/beans_resnet18.pth")
